In [ ]:
from pyspark import SparkConf, SparkContext
from prettytable import PrettyTable

# Instead of doing sparkContext, we can set master in the config.  
conf = SparkConf().setMaster('local[*]').setAppName('WordCount')

# Then just pass it to our context.  
sc = SparkContext(conf = conf)

# Read inn a text file.  Same directory, easy read.  
book = sc.textFile('./pride_and_prejudice.txt')

words = book.flatMap(lambda x: x.split())

# Get the dictionary of words- all unique words and the number of occurences.  
word_counts = words.countByValue()

table = PrettyTable()
table.field_names = ['Word', 'Count']


for word, count in word_counts.items():
    table.add_row([word, count])

print(table)

Improved word count collection.  Filter out other characters

In [ ]:
from pyspark import SparkConf, SparkContext
from prettytable import PrettyTable
import re

# Instead of doing sparkContext, we can set master in the config.  
conf = SparkConf().setMaster('local[*]').setAppName('WordCount')

# Then just pass it to our context.  
sc = SparkContext(conf = conf)

# Reusable regular expression.  Find beginning \b of words \w, don't capture apostrophes
WORD_RE = re.compile(r"\b\w+(?: '\w+)*\b", re.UNICODE)

def normalizeWords(line):
    return WORD_RE.findall(line.lower())

# Read inn a text file.  Same directory, easy read.  
book = sc.textFile('./pride_and_prejudice.txt')

words = book.flatMap(normalizeWords)

# Get the dictionary of words- all unique words and the number of occurences, as a tuple of the value and 1.  
word_counts = words.map(lambda x: (x, 1)).reduceByKey(lambda a, b: a + b) # Reduce by key is sort of like group by, for word a, key b in this case.  
sorted_counts = word_counts.sortBy(lambda x: x[1], ascending=False) # Sort each line x by the second column.  (The count column)

table = PrettyTable()
table.field_names = ['Word', 'Count']


for word, count in sorted_counts.collect():
    table.add_row([word, count])

print(table)

sc.stop()

This next file gets the total order value for each customer.  
Total order value by customer.  
Customer ID:  Then total of all their orders by price on the right.  Who spent the most?  

In [ ]:
from pyspark import SparkConf, SparkContext
from prettytable import PrettyTable

conf = SparkConf().setMaster('local[*]').setAppName('WordCount')

sc = SparkContext(conf = conf)


# Get all orders.  
orders = sc.textFile('./customer-orders.csv')

# Now we need to get rid of the order id for each order, since we only care about the total amount per customer.  
def filter_orders(line):
    traits = line.split(',') # CSV.  
    customer_id = traits[0]
    price = float(traits[2])
    return (customer_id, price)


# Then, for each line, we map a line to a key value RDD.  
order_pairs = orders.map(filter_orders) # This 
grouped_orders = order_pairs.reduceByKey(lambda a, b: a + b) # For each matching key, reduce two into one value as a total

sorted_orders = grouped_orders.sortBy(lambda x: x[1], ascending=False)

table = PrettyTable()
table.field_names = ['Customer ID', 'Total Spent']

for customer, total in sorted_orders.collect():
    table.add_row([customer, total])

print(table)

sc.stop()
